# Spatial accuracy analysis

In this notebook we analyze the spatial accuracy of detected transcripts for various spatial transcriptomics platforms assessed in our benchmark study. We focus on transcripts for the Ptgds gene in the mouse brain tissue sections. There Ptgds transcripts are present in a thin cell layer in the tissue boundary.


## Python imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import gc
import spatialdata as sd
from diffusion import LevelInfo, Experiment
from diffusion_plots import (
    DiffusionPlotSettings,
    plot_shapely_geometries,
    plot_tissue_outline_on_HE_image,
    plot_bins_colored_by_feature,
    plot_distance_kde_combined,
    plot_distance_kde_ridgeline,
    plot_distance_cdf,
)
from diffusion_utils import (
    compute_diffusion,
    generate_diffusion_plots,
    print_geodataframe_geometry_info,
)

/data/groups/technologies/spatial.catalyst/Frank/2024-07-UTBenchmark-SpC/diffusion/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Experiments definition

Info about experiments for which to perform spatial accuracy analysis.

In [3]:
SOURCE_PATH = "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/"  # do *NOT* use the .zarr files in here directly
WORK_PATH = "/data/groups/technologies/spatial.catalyst/Frank/2024-07-UTBenchmark-SpC/diffusion"
SDATA_PATH = os.path.join(WORK_PATH, "data")
PLOTS_PATH = os.path.join(WORK_PATH, "plots")

In [4]:
# Choose boundary sections far enough apart that their eligible analysis bins are disjoint.
# Distances/counts are pooled per experiment without overlap detection or deduplication.
# Exactly identical endpoints, e.g. ((1000, 2000), (1000, 2000)), select the
# full nearest exterior ring in its stored order. Hole rings are excluded.
# Different boundary point pairs may select different geometries; each pair's endpoints must
# be nearest to the same exterior contour.
experiments = [
    Experiment("BMKMANU S1000",
               "Mouse brain",
               "exp74",
               "BMK_S1000/exp74/harpy/sdata.zarr",
               "exp74_he",
               0.341,  # Pixel size: 0.341 µm
               "exp74_tissue",
               [LevelInfo(f"level_{level}", f"exp74_level_{level}_table", f"exp74_level_{level}_labels") for level in [1, 2, 3, 4, 5, 6, 7, 13]],
               LevelInfo("hexagonal_grid_20um", "hexagonal_grid_20um_table", "hexagonal_grid_20um_labels"),
               boundary_point_pairs=[((15000, 16000), (18000, 8000)),
                                     ((1500, 6000), (2500, 9500))]),   
    Experiment("BMKMANU S3000",
               "Mouse brain",
               "exp99",
               "BMK_S3000/Exp99/harpy/sdata.zarr",
               "exp99_he",
               0.340,  # Pixel size: 0.340 µm
               "exp99_tissue", 
               [LevelInfo(f"exp99_level_{level}", f"exp99_level_{level}_table", f"exp99_level_{level}_labels") for level in [1,2,3,4,5,6,7,9,18]],
               LevelInfo("hexagonal_grid_20um", "hexagonal_grid_20um_table", "hexagonal_grid_20um_labels"),
               boundary_point_pairs=[((16000, 5000), (11500, 17000))]),  
    Experiment("Nova-ST",
               "Mouse brain",
               "Exp65-sampleSPC014",
               "Nova-ST/Exp65-sampleSPC014/harpy/sdata.zarr",
               "Exp65-sampleSPC014_image",
               1.000,  # Pixel size: 1.000 µm
               "Exp65-sampleSPC014_tissue", 
               [LevelInfo("table", "table", "Exp65-sampleSPC014_labels")],
               LevelInfo("hexagonal_grid_20um", "hexagonal_grid_20um_table", "hexagonal_grid_20um_labels"),
               boundary_point_pairs=[((5850, 6000), (4500, 1100))]), 
    Experiment("Visium HD (FFPE)",
               "Mouse brain",
               "SCA003",
               "Visium_HD/SCA003/harpy/sdata.zarr",
               "SCA003_full_image",
               0.442,  # Pixel size: 0.442 µm
               "SCA003_tissue",
               [LevelInfo(f"square_{bin_size:03}um", f"square_{bin_size:03}um", f"SCA003_square_{bin_size:03}um_labels") for bin_size in [2, 8, 16, 32]],
               LevelInfo("hexagonal_grid_20um", "hexagonal_grid_20um_table", "hexagonal_grid_20um_labels"),
               boundary_point_pairs=[((2200, 9200), (11000, 13500))]), 
    Experiment("Visium HD (FF)",
               "Mouse brain",
               "SCA019",
               "Visium_HD/SCA019/harpy/sdata.zarr",
               "SCA019_full_image",
               0.442,  # Pixel size: 0.442 µm
               "SCA019_tissue", 
               [LevelInfo(f"square_{bin_size:03}um", f"square_{bin_size:03}um", f"SCA019_square_{bin_size:03}um_labels") for bin_size in [2, 8, 16, 32]],
               LevelInfo("hexagonal_grid_20um", "hexagonal_grid_20um_table", "hexagonal_grid_20um_labels"),
               boundary_point_pairs=[((4500, 2500), (1900, 11900)),  # top left section
                                     ((2800, 13000), (7500, 15500))]),  # bottom left section
    Experiment("Visium HD 3'",
               "Mouse brain",
               "SCA024",
               "Visium_HD/SCA024/harpy/sdata.zarr",
               "SCA024_full_image",
               0.442,  # Pixel size: 0.442 µm
               "SCA024_tissue",
               [LevelInfo(f"square_{bin_size:03}um", f"square_{bin_size:03}um", f"SCA024_square_{bin_size:03}um_labels") for bin_size in [2, 8, 16, 32]],
               LevelInfo("hexagonal_grid_20um", "hexagonal_grid_20um_table", "hexagonal_grid_20um_labels"),
               boundary_point_pairs=[((5900, 700), (2200, 8500)),  # left section
                                     ((9000, 9100), (7000, 700))]),  # right section
    Experiment("Stereo-Seq",
               "Mouse brain",
               "spc014",
               "Stereo-Seq/spc014/harpy/sdata.zarr",
               "spc014_image",
               0.500,  # Pixel size
               "spc014_tissue", 
               [LevelInfo("spc014_raw_table", "spc014_raw_table", "spc014_raw_labels")],
               LevelInfo("hexagonal_grid_20um", "hexagonal_grid_20um_table", "hexagonal_grid_20um_labels"),
               boundary_point_pairs=[((14500, 12000), (6000, 15000))]),
]

# NOTE:
# No spatial accuracy analysis was performed for these platforms:
# - Visium FFPE: distance between capture spots too large
# - Curio Seeker: no microscope images available + tissue border (with Ptgds cell layer) outside capture area

In [5]:
GENE_NAME = "Ptgds"  # We will use this gene for spatial accuracy analysis across datasets, it is expressed on the brain tissue boundary

MAX_DIFFUSION_DISTANCE_UM = 200.0  # transcripts within this distance from tissue boundary will be analyzed

## Subset SpatialData objects

To avoid overwriting the SpatialData objects, we extract the subsets that we need and write them as new SpatialData objects.
These new objects will then be analyzed for spatial accuracy.

In [6]:
MAKE_SUBSETTED_SDATA = False

if MAKE_SUBSETTED_SDATA:
    os.makedirs(SDATA_PATH, exist_ok=True)

    for experiment in experiments:
        # Read original sdata
        orig_zarr_path = os.path.join(SOURCE_PATH, experiment.sdata_path)
        print(f'Reading {orig_zarr_path}')
        sdata = sd.read_zarr(orig_zarr_path)
        print(sdata)

        # Copy the sdata object (to avoid overwriting it during spatial accuracy analysis).
        # Keep only elements that are relevant for spatial accuracy analysis while we're at it.
        keep_elements = [experiment.he_image, 
                         experiment.tissue_mask] + \
                        [experiment.hexbins.labels_layer,
                         experiment.hexbins.table_name] + \
                        [level.labels_layer for level in experiment.levels] + \
                        [level.table_name for level in experiment.levels] 

        print (f'Subsetting sdata -- keep elements {keep_elements}')

        sdata_subset = sdata.subset(
            element_names=keep_elements,
            filter_tables=False,
        )
        print(sdata_subset)

        # For some datasets, the tissue is fragmented and the tissue mask consists of disjunct regions.
        # These regions are stored as separate Polygons (rows) in the shapes layer's GeoDataframe,
        # but they happen to have the same index in the Dataframe. This identical index confuses the function _validate_polygons()
        # in spatialdata_plot (v0.2.14) and leads to only one Polygon being plotted.
        # Avoid this plotting problem by making the index unique here.
        if experiment.tissue_mask is not None:
            sdata_subset.shapes[experiment.tissue_mask].reset_index(inplace=True)

        # Write subsetted sdata to disk
        work_zarr_path = os.path.join(SDATA_PATH, experiment.sdata_path)
        print(f'Writing {work_zarr_path}')
        sdata_subset.write(work_zarr_path, overwrite=True)


## Closer look at the tissue outlines

Inspect how many disjunct polygons the tissue outline consists of. 

In [7]:
PLOT_CONTOUR_INFO = False

if PLOT_CONTOUR_INFO: 
    for experiment in experiments:
        print(f">>> {experiment.technology} {experiment.dataset_id} <<<")

        zarr_path = os.path.join(SDATA_PATH, experiment.sdata_path)
        print(f"Reading {zarr_path}")
        sdata = sd.read_zarr(zarr_path)

        output_folder = os.path.join(PLOTS_PATH, experiment.technology, experiment.dataset_id, "overview")
        os.makedirs(output_folder, exist_ok=True)

        print_geodataframe_geometry_info(sdata, experiment.tissue_mask)

        plot_shapely_geometries(sdata,
                                experiment.tissue_mask,
                                output_path=os.path.join(output_folder, "tissue_mask.png"),
                                title=f"{experiment.technology}\nTissue mask")

## Overview images

In [8]:
PLOT_OVERVIEW_INFO = False

if PLOT_OVERVIEW_INFO:
    for experiment in experiments:
        print(f">>> {experiment.technology} {experiment.dataset_id} <<<")
        
        he_image = experiment.he_image
        tissue_mask = experiment.tissue_mask

        zarr_path = os.path.join(SDATA_PATH, experiment.sdata_path)
        print(f"Reading {zarr_path}")
        sdata = sd.read_zarr(zarr_path)
        print(sdata)

        output_folder = os.path.join(PLOTS_PATH, experiment.technology, experiment.dataset_id, "overview")
        os.makedirs(output_folder, exist_ok=True)

        if he_image is not None and tissue_mask is not None:
            gc.collect()
            filename = 'tissue.png'
            print(experiment)
            print(experiment.pixel_size_um)
            plot_tissue_outline_on_HE_image(sdata,
                                            he_image,
                                            tissue_mask,
                                            experiment.technology,
                                            experiment.dataset_id,
                                            experiment.pixel_size_um,
                                            os.path.join(output_folder, filename),
                                            fill_alpha=0.0,
                                            outline_width=1)

        gc.collect()
        print(f"> Adata for {experiment.hexbins}:")
        adata = sdata.tables[experiment.hexbins.table_name]
        print(adata)

        print(f"> {GENE_NAME} bins for {experiment.hexbins}")
        plot_bins_colored_by_feature(sdata,
                                     he_image,
                                     experiment.technology,
                                     experiment.dataset_id,
                                     experiment.pixel_size_um,
                                     experiment.hexbins,
                                     GENE_NAME,
                                     os.path.join(output_folder, f'{experiment.hexbins.name}_{GENE_NAME}.png'))
        
        for level in experiment.levels:
            gc.collect()
            print(f"> Adata for {level}:")
            adata = sdata.tables[level.table_name]
            print(adata)

            print(f"> {GENE_NAME} bins for {level}")
            plot_bins_colored_by_feature(sdata,
                                         he_image,
                                         experiment.technology,
                                         experiment.dataset_id,
                                         experiment.pixel_size_um,
                                         level,
                                         GENE_NAME,
                                         os.path.join(output_folder, f'{level.name}_{GENE_NAME}.png'))

## Spatial accuracy analysis per technology

In [ ]:
COMPUTE_DIFFUSION = True
GENERATE_DIFFUSION_PLOTS = True

CROP_MARGIN_PX = 200.0  # Visual margin for plots, irrelevant for analysis
MAX_INDIVIDUAL_PLOT_DIFFUSION_DISTANCE_UM = 100.0  # for plotting, close-up
BIN_WIDTHS_UM = [1.0, 2.0, 3.0, 4.0, 5.0, 10.0, 20.0]

if COMPUTE_DIFFUSION or GENERATE_DIFFUSION_PLOTS:
    for experiment in experiments:
        region_specs = experiment.boundary_point_pairs
        if region_specs:
            finest_level = experiment.levels[0]  # first level is the finest
            
            plot_settings = DiffusionPlotSettings(
                histogram_bin_widths_um=BIN_WIDTHS_UM,
                max_plot_distance_um=MAX_INDIVIDUAL_PLOT_DIFFUSION_DISTANCE_UM,
                tissue_outline=True,               # -> analysis/tissue.png (all sections)
                boundary=False,                    # -> analysis/crop{i}/boundary.png
                bin_gene_counts=False,             # -> analysis/crop{i}/transcripts_closeup.png
                full_tissue_bin_gene_counts=True,  # -> analysis/transcripts.png (all sections)
                bins_and_closest_points=False,     # -> analysis/crop{i}/closest_points.png
                distances_linearized=False,        # -> analysis/crop{i}/distances_linearized
                histogram=True,                    # -> analysis/pooled/histogram_binwidth_{bw}um.png
                boxplot=False                      # -> analysis/pooled/boxplot_um.png
            )

            max_diffusion_distance_px = MAX_DIFFUSION_DISTANCE_UM / experiment.pixel_size_um

            if COMPUTE_DIFFUSION:
                compute_diffusion(SDATA_PATH,
                                  experiment,
                                  level=finest_level.table_name,
                                  gene_name=GENE_NAME,
                                  plots_folder=PLOTS_PATH,
                                  max_distance_px=max_diffusion_distance_px,
                                  crop_margin_px=CROP_MARGIN_PX,
                                  use_signed_distances=True)   # calculate *signed* distances (left/right of tissue boundary, inside/outside of tissue)

            if GENERATE_DIFFUSION_PLOTS:
                generate_diffusion_plots(SDATA_PATH,
                                         experiment,
                                         level=finest_level.table_name,
                                         plots_folder=PLOTS_PATH,
                                         plot_settings=plot_settings)
        

## Spatial accuracy analysis summary

Now generate some plots that compare the spatial accuracy data (distances of transcripts to boundary) for the different platforms.

In [12]:
MAX_COMBINED_PLOT_DIFFUSION_DISTANCE_UM = 50.0

In [13]:
combined_dir = os.path.join(PLOTS_PATH, "combined")
os.makedirs(combined_dir, exist_ok=True)

In [ ]:
crop_specs = []
for experiment in experiments:   
    region_specs = experiment.boundary_point_pairs
    for i in range(len(region_specs)):
        crop_spec = (
            experiment.technology,
            experiment.dataset_id,
            experiment.pixel_size_um,
            f"crop{i+1}",
        )
        crop_specs.append(crop_spec)
print(crop_specs)

[('BMKMANU S1000', 'exp74', 0.341, 'crop1'), ('BMKMANU S1000', 'exp74', 0.341, 'crop2'), ('BMKMANU S3000', 'exp99', 0.34, 'crop1'), ('Nova-ST', 'Exp65-sampleSPC014', 1.0, 'crop1'), ('Visium HD (FFPE)', 'SCA003', 0.442, 'crop1'), ('Visium HD (FF)', 'SCA019', 0.442, 'crop1'), ('Visium HD (FF)', 'SCA019', 0.442, 'crop2'), ("Visium HD 3'", 'SCA024', 0.442, 'crop1'), ("Visium HD 3'", 'SCA024', 0.442, 'crop2'), ('Stereo-Seq', 'spc014', 0.5, 'crop1')]


### Histograms and ridgeline plots

In [15]:
for bin_width_um in BIN_WIDTHS_UM:
    bin_width_str = f"binwidth_{bin_width_um:g}um".replace(".", "p")
    print(f"Bin width {bin_width_um} um")

    output_path=os.path.join(PLOTS_PATH, "combined", bin_width_str, "distance_kde_um.png")
    plot_distance_kde_combined(
        crop_specs,
        output_path,
        PLOTS_PATH,
        GENE_NAME,
        MAX_COMBINED_PLOT_DIFFUSION_DISTANCE_UM,
        align_peaks=False,
        bin_width_um=bin_width_um,
    )

    output_path=os.path.join(PLOTS_PATH, "combined", bin_width_str, "distance_kde_um_peaksaligned.png")
    plot_distance_kde_combined(
        crop_specs,
        output_path,
        PLOTS_PATH,
        GENE_NAME,
        MAX_COMBINED_PLOT_DIFFUSION_DISTANCE_UM,
        align_peaks=True,
        bin_width_um=bin_width_um,
    )

    output_path=os.path.join(PLOTS_PATH, "combined", bin_width_str, "distance_kde_ridgeline_um_peaksaligned.png")
    plot_distance_kde_ridgeline(
        crop_specs,
        output_path,
        PLOTS_PATH,
        GENE_NAME,
        MAX_COMBINED_PLOT_DIFFUSION_DISTANCE_UM,
        align_peaks=True,
        overlap=0.97,
    )

Bin width 1.0 um
Using shared KDE bandwidth: 1.560 µm for BMKMANU S1000, BMKMANU S3000, Nova-ST, Visium HD (FFPE), Visium HD (FF), Visium HD 3', Stereo-Seq
Using shared KDE bandwidth: 1.560 µm for BMKMANU S1000, BMKMANU S3000, Nova-ST, Visium HD (FFPE), Visium HD (FF), Visium HD 3', Stereo-Seq
Using shared KDE bandwidth: 1.560 µm for BMKMANU S1000, BMKMANU S3000, Nova-ST, Visium HD (FFPE), Visium HD (FF), Visium HD 3', Stereo-Seq
Bin width 2.0 um
Using shared KDE bandwidth: 1.560 µm for BMKMANU S1000, BMKMANU S3000, Nova-ST, Visium HD (FFPE), Visium HD (FF), Visium HD 3', Stereo-Seq
Using shared KDE bandwidth: 1.560 µm for BMKMANU S1000, BMKMANU S3000, Nova-ST, Visium HD (FFPE), Visium HD (FF), Visium HD 3', Stereo-Seq
Using shared KDE bandwidth: 1.560 µm for BMKMANU S1000, BMKMANU S3000, Nova-ST, Visium HD (FFPE), Visium HD (FF), Visium HD 3', Stereo-Seq
Bin width 3.0 um
Using shared KDE bandwidth: 1.560 µm for BMKMANU S1000, BMKMANU S3000, Nova-ST, Visium HD (FFPE), Visium HD (FF), V

### Cumulative distribution function (CDF) plot

In [16]:
plot_distance_cdf(
    crop_specs=crop_specs,
    output_path=os.path.join(combined_dir, "cdf_distance.png"),
    plots_root=PLOTS_PATH,
    max_distance_um=MAX_COMBINED_PLOT_DIFFUSION_DISTANCE_UM,
    origin="peak",
)

Using shared KDE bandwidth: 1.560 µm for BMKMANU S1000, BMKMANU S3000, Nova-ST, Visium HD (FFPE), Visium HD (FF), Visium HD 3', Stereo-Seq
Saved /data/groups/technologies/spatial.catalyst/Frank/2024-07-UTBenchmark-SpC/diffusion/plots/combined/cdf_distance.png
